# Light ROI extraction: K-means: scores at full size (BTXRD)
Measures PSNR, SSIM and CII of K-means on the **full-size** X-rays (2440 x 2440), before the images are made smaller.
It uses exactly the same settings and function as `kmeans_btxrd.ipynb`. The scores in that notebook were measured on the 512 x 512 images; these are the same scores at full size.
Nothing is saved: every image is only kept in memory while it is measured. The original data is only read, never changed.

## Settings

In [1]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
from sklearn.cluster import KMeans
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

DATA = Path("../../../../data/BTXRD no duplicates")    # the original full-size images without duplicates (only read)
ANNOTATIONS = DATA / "Annotations"                     # tumor outlines (only read)
SPLITS = ["train", "validation", "test"]

CLUSTERS = 3         # K-means: the gray values of the X-ray are sorted into 3 groups (dark, middle, bright)
CLOSE = 25           # mask clean-up: closes gaps up to 25 pixels wide
OPEN = 25            # mask clean-up: removes specks thinner than 25 pixels
MARGIN = 50          # mask clean-up: 50 pixels extra on every side (about 10 pixels on the 512 x 512 image)
BLACK = 15           # pixels at or below this are black background
RING = 48            # CII: width of the ring around the tumor, in pixels on the full image (10 pixels on 512 x 512 = 10 x 2440 / 512)

## List the images
Every image in the `train`, `validation` and `test` folders, with its label (normal or tumor). The same images as in the preprocessing notebook.

In [2]:
records = []
for split in SPLITS:
    for path in sorted((DATA / split).glob("*/*.jp*g")):       # */ = normal or tumor, files end in .jpeg or .jpg
        records.append({"image": path.stem, "split": split, "label": path.parent.name, "path": path})

images = pd.DataFrame(records)
print(len(images), "images")
print(images.groupby(["split", "label"]).size())

3723 images
split       label 
test        normal     280
            tumor      279
train       normal    1306
            tumor     1300
validation  normal     280
            tumor      278
dtype: int64


## Functions
Load an image at full size, apply K-means (copied from `kmeans_btxrd.ipynb`), and find the X-ray inside the black square.

In [3]:
def load(path):
    image = Image.open(path).convert("L")                      # gray, pixels 0 (black) to 255 (white)
    return np.array(image)                                     # full size: 2440 x 2440


def disk(size):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))   # a round shape, size pixels wide


def clean(mask):
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, disk(CLOSE))   # close small gaps
    mask = ndimage.binary_fill_holes(mask).astype(np.uint8)   # fill holes, so a dark part inside a bone (like a lytic tumor) stays in
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, disk(OPEN))  # remove small specks
    return cv2.dilate(mask, disk(MARGIN)) == 1                 # a safety margin around the mask; True = keep


def kmeans_roi(image):
    xray = image[image > BLACK]                                # the X-ray pixels, not the black square
    counts = np.bincount(xray, minlength=256)                  # how many pixels have each gray value
    values = np.nonzero(counts)[0]                             # the gray values that occur
    kmeans = KMeans(n_clusters=CLUSTERS, n_init=10, random_state=42)   # random_state: the same result every time
    kmeans.fit(values.reshape(-1, 1).astype(np.float64), sample_weight=counts[values])   # each gray value counts as often as it occurs
    centers = np.sort(kmeans.cluster_centers_.ravel())         # the middle gray value of each group, from dark to bright
    level = (centers[0] + centers[1]) / 2                      # the border between the darkest group and the next one
    mask = clean(image > level)                                # everything except the darkest group (the body), cleaned up
    return np.where(mask, image, 0).astype(np.uint8)           # keep the X-ray inside the mask, black outside


def xray_box(image):
    rows = np.where((image > BLACK).any(axis=1))[0]            # rows with something brighter than black
    cols = np.where((image > BLACK).any(axis=0))[0]            # the same for columns
    return rows[0], rows[-1] + 1, cols[0], cols[-1] + 1        # top, bottom, left, right

## Measuring functions
Both images are compared at full size (2440 x 2440): the original X-ray (before) and the X-ray after K-means (after).
- PSNR and SSIM: how much the image changed (higher = less change). Measured on the X-ray only, not the black square.
- CII: how much more the tumor stands out from the ring around it. Above 1 = it stands out more. The ring is 48 pixels wide here, the same area as the 10-pixel ring on the 512 x 512 image.
Note for CII: where the mask makes part of the ring around a tumor black, CII changes too, so CII says less here than for the other methods.

In [4]:
def tumor_mask(name, size):
    annotation = json.loads((ANNOTATIONS / (name + ".json")).read_text())
    mask = np.zeros(size, dtype=np.uint8)                      # full size: the outline was drawn on the 2440 x 2440 image
    for shape in annotation["shapes"]:
        if shape["shape_type"] == "polygon":                   # the outline (the rectangles are only boxes)
            points = np.round(np.array(shape["points"])).astype(np.int32)
            cv2.fillPoly(mask, [points], 1)
    return mask == 1                                           # True inside the tumor


def ring_around(tumor, before):
    kernel = np.ones((2 * RING + 1, 2 * RING + 1), dtype=np.uint8)
    bigger = cv2.dilate(tumor.astype(np.uint8), kernel) == 1   # the tumor made RING pixels bigger
    return bigger & ~tumor & (before > BLACK)                  # the ring: not the tumor, not the black background


def contrast(image, tumor, ring):
    t = image[tumor].mean()                                    # tumor brightness
    r = image[ring].mean()                                     # ring brightness
    return abs(t - r) / (t + r)                                # abs: a tumor can be darker or brighter than the bone


def cii(before, after, tumor, ring):
    if tumor.sum() == 0 or ring.sum() == 0:
        return np.nan                                          # cannot be measured
    contrast_before = contrast(before, tumor, ring)
    if contrast_before == 0:
        return np.nan                                          # cannot divide by 0
    return contrast(after, tumor, ring) / contrast_before


def psnr_ssim(before, after):
    top, bottom, left, right = xray_box(before)                # the X-ray only
    a, b = before[top:bottom, left:right], after[top:bottom, left:right]
    return peak_signal_noise_ratio(a, b, data_range=255), structural_similarity(a, b, data_range=255)

## Measure every image
For every image: K-means on the full image, then PSNR, SSIM and CII against the original full image. CII is only for tumor images. Nothing is saved. This takes some minutes.

In [5]:
rows = []
for n, (_, item) in enumerate(images.iterrows(), start=1):
    before = load(item["path"])                                # the original X-ray at full size
    after = kmeans_roi(before)                                 # K-means at full size (only in memory, not saved)

    psnr, ssim = psnr_ssim(before, after)
    row = {"image": item["image"], "label": item["label"], "PSNR": psnr, "SSIM": ssim, "CII": np.nan}
    if item["label"] == "tumor":
        tumor = tumor_mask(item["image"], before.shape)
        row["CII"] = cii(before, after, tumor, ring_around(tumor, before))
    rows.append(row)

    if n % 500 == 0:
        print(n, "of", len(images), "done")

scores = pd.DataFrame(rows)
print(len(scores), "images measured")

KeyboardInterrupt: 

## Results
The middle value (median) of each score at full size: the image after K-means compared with the original image. CII only counts the tumor images.

In [ ]:
scores[["PSNR", "SSIM", "CII"]].median().round(3)